# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()

print("\033[1mDataset name:\033[0m", metadata_json.get('name', 'N/A'))
print("\033[1mDescription:\033[0m", metadata_json.get('description', 'N/A'))

## 2. Data Overview
Review available record sets, fields, and their IDs. Use the `@id` references for each entity.

In [ ]:
# Retrieve all record sets by @id
record_sets = dataset.metadata.record_sets

if not record_sets:
    print("No record sets found in the Croissant metadata.")
else:
    print(f"Found {len(record_sets)} record set(s):\n")
    for rs in record_sets:
        print(f"Record Set @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', '')}")
        print(f"  Description: {rs.get('description', '')}")        fields = rs.get('field', [])        # Ensure fields is a list        if isinstance(fields, dict):
            fields = [fields]
        if fields:
            print("  Fields:")            for field in fields:
                if isinstance(field, dict):
                    print(f"    - @id: {field.get('@id')} Name: {field.get('name','')}")
                else:
                    print(f"    - @id: {field}")        print("")# If record sets are not present in metadata, skip to manual record set id (see next cell).

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

_Note: If the dataset metadata does not return record sets in the previous cell, you may need to inspect the schema or documentation for available record set IDs. Based on prior inspection, we set the record set id below._

In [ ]:
# Example: Inspect Croissant JSON metadata for record set @ids
import requests
schema = requests.get(croissant_url).json()

# Try to extract the recordSet @ids from @graph
record_set_ids = []
for entity in schema.get("@graph", []):
    if entity.get("@type") == "cr:RecordSet" or entity.get("@type") == "RecordSet":
        record_set_ids.append(entity["@id"])

print("RecordSet @ids found in the graph:")
print(record_set_ids)

# For this dataset, we'll use the first record set @id found (if any)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
else:
    # Manually set based on knowledge of FAIR2 datasets (example):
    main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/ee0b683a-cb1c-4682-8d78-b94f0b9c440f' # Replace with the correct @id from schema if found
print(f"\nSelected record set @id: {main_record_set_id}")

In [ ]:
# Load records for all (or a selection of) record sets
dataframes = {}

# For simplicity, we use only the main record set id for demonstration (could add more)
print(f"Loading records for record set: {main_record_set_id}")
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
dataframes[main_record_set_id] = df

print("Columns in the DataFrame:")
print(df.columns.tolist())

print("\nPreview:")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a numeric field for demonstration
# Printing columns to help us pick one
print("Columns available:", df.columns.tolist())

# Let's assume the dataset contains a column like 'cr:mw' (molecular weight), or similar
candidate_numeric_fields = [c for c in df.columns if 'mw' in c.lower() or 'abundance' in c.lower() or 'count' in c.lower()]
if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]
    print(f"Using numeric field: {numeric_field}")
else:
    numeric_field = df.select_dtypes('number').columns[0]
    print(f"Defaulting to first numeric column: {numeric_field}")

# Set thresholds and group field
threshold = df[numeric_field].mean() if df[numeric_field].dtype != 'O' else 10

# Try to get a group field (e.g. protein description, if present)
candidate_group_fields = [c for c in df.columns if 'desc' in c.lower() or 'group' in c.lower() or 'type' in c.lower() or 'sample' in c.lower()]
if candidate_group_fields:
    group_field = candidate_group_fields[0]
    print(f"Grouping by: {group_field}")
else:
    group_field = None

# Basic filtering and normalization using numeric_field
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped by {group_field} (mean of {numeric_field}):")
        print(grouped_df.head())
else:
    print(f"{numeric_field} is not numeric; skipping numeric EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Plot histograms and relationships if fields are appropriate
import matplotlib.pyplot as plt
import seaborn as sns

if pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If group_field is categorical with limited unique values, show boxplot
    if group_field and df[group_field].nunique() < 20:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded and explored the FAIR^2 dataset using the Croissant schema with `mlcroissant`.
- Record sets and fields were inspected using their unique `@id` identifiers, ensuring reproducibility.
- A subset of numeric features was selected for exploratory and visualization purposes, demonstrating filtering and normalization.
- This workflow can be extended for in-depth domain analysis, modeling, or FAIR^2 data curation tasks.